In [2]:
import polars as pl

### Load the OJPs datasets

In [3]:
ojp = pl.read_parquet("../data/lightcast/ITA_2025_postings.parquet")
ojp = ojp.rename({col: col.lower().replace(" ", "_") for col in ojp.columns})
#ojp_pd = pd.read_csv("../data/lightcast/ITA_2025_postings.csv")

# Only keep postings in Calabria (ITF6)
ojp_calabria_coded = ojp.filter(pl.col("laa_admin_area_1") == "ITF6")
del ojp

print(f"Number of rows: {ojp_calabria_coded.height:,} (number of postings in Calabria)")

Number of rows: 27,012 (number of postings in Calabria)


In [4]:
# Drop Undefined (99) and Volunteer (29) LOT Career Areas
ojp_calabria_coded = ojp_calabria_coded.filter(
    ~pl.col('lot_career_area').is_in(['99', '29'])
).unique()

print(f"Number of rows: {ojp_calabria_coded.height:,} (number of postings in Calabria after dropping undefined and volunteer LOT Career Areas)")

Number of rows: 26,137 (number of postings in Calabria after dropping undefined and volunteer LOT Career Areas)


### Decode OJP dataset using LightCast taxonomy dictionaries

In [5]:
ojp_calabria = ojp_calabria_coded.clone()

# Get all columns that match the LOT_* pattern
lot_columns = [col for col in ojp_calabria.columns if col.startswith(('laa_', 'lot_', 'naics_', 'isco_'))]

for col_name in lot_columns:
    # Auto-generate dict_name and name_col based on pattern
    dict_name = col_name.lower()  # lot_career_area -> lot_career_area
    name_col = f"{col_name}_name"  # lot_career_area -> lot_career_area_name
    code_col = f"{col_name}_code"  # lot_career_area -> lot_career_area_code
    
    # Load dictionary and normalize column names
    dict_df = pl.read_parquet(f"../data/lightcast/dictionaries/{dict_name}_dictionary.parquet")
    dict_df = dict_df.rename({col: col.lower().replace(" ", "_") for col in dict_df.columns})
    
    # Get original column order
    original_columns = ojp_calabria.columns
    col_position = original_columns.index(col_name)
    
    # Join to add the name column
    ojp_calabria = ojp_calabria.join(
        dict_df.select([col_name, name_col]),
        on=col_name,
        how='left'
    ).rename({col_name: code_col})
    
    # Reorder columns: insert CODE and NAME together at original position
    current_columns = ojp_calabria.columns
    other_columns = [c for c in current_columns if c not in [code_col, name_col]]
    new_order = other_columns[:col_position] + [code_col, name_col] + other_columns[col_position:]
    ojp_calabria = ojp_calabria.select(new_order)
    
    del dict_df

del new_order, lot_columns, original_columns, other_columns, current_columns
ojp_calabria = ojp_calabria.unique("id")

print(f"Number of rows: {ojp_calabria.height:,}")

Number of rows: 26,137


### Map CP2021 occupations (from Training Catalogue) to ISCO-08 occupations (used in LightCast)

In [6]:
# cp2021_isco08_crosswalk = pl.read_csv("../data/isco08_cp2021_crosswalk.csv", schema_overrides={'isco_08_03_code': pl.String}, separator=";")
# cp2021_isco08_crosswalk = cp2021_isco08_crosswalk.rename({col: col.lower().replace(" ", "_") for col in cp2021_isco08_crosswalk.columns})

# # Join cp2021_isco08_crosswalk to ojp_calabria on isco_08_3_code column
# ojp_calabria = ojp_calabria.join(
#     cp2021_isco08_crosswalk.select(['isco_08_03_code', 'cp2021_code', 'cp2021_name']),
#     left_on='isco_08_3_code',
#     right_on='isco_08_03_code',
#     how='left'
# )

### Load the skills datasets

In [7]:
skills_all = pl.read_parquet("../data/lightcast/ITA_2025_skills.parquet")
skills_all = skills_all.rename({col: col.lower().replace(" ", "_") for col in skills_all.columns})

skills_dict = pl.read_parquet("../data/lightcast/dictionaries/skill_id_dictionary.parquet")
skills_dict = skills_dict.rename({col: col.lower().replace(" ", "_") for col in skills_dict.columns})

print(f"Number of rows: {skills_all.height:,} (number of skills listed in all 2025 postings worldwide)")

Number of rows: 25,640,877 (number of skills listed in all 2025 postings worldwide)


In [8]:
full_skill_list = pl.read_csv("../data/lightcast/dictionaries/full_skill_list.csv", separator=";")
full_skill_list = full_skill_list.rename({col: col.lower().replace(" ", "_") for col in full_skill_list.columns})

print(f"Number of rows: {full_skill_list.height:,} (number of unique skills in LightCast taxonomy)")

Number of rows: 34,639 (number of unique skills in LightCast taxonomy)


In [9]:
# Map skills names to skill IDs

skills_all = (skills_all.join(skills_dict, on='skill_id', how='left')
          .select(['id', 'skill_name', 'skill_id'] + [col for col in skills_all.columns if col not in ['id', 'skill_name', 'skill_id']]))
del skills_dict

### Filter the skills that appear in Calabria OJPs, drop the rest

In [10]:
# Get unique IDs from ojp_calabria
calabria_ids = ojp_calabria.select("id").unique()

# Filter skills_all lazily
skills_calabria = skills_all.lazy().join(
    calabria_ids.lazy(),
    on="id",
    how="inner"
).collect()

del skills_all
del calabria_ids

print(f"Number of rows: {skills_calabria.height:,} (number of skills mentioned in postings in Calabria)")

Number of rows: 168,923 (number of skills mentioned in postings in Calabria)


### Decode Skills dataset using LightCast skill taxonomy dictionaries

In [11]:
# Get all skill taxonomy columns that need to be mapped
skill_columns = [col for col in skills_calabria.columns if col.startswith(('skill_category', 'skill_subcategory'))]

for col_name in skill_columns:
    # Auto-generate dict_name and name_col based on pattern
    dict_name = col_name.lower()  # skill_category -> skill_category
    name_col = f"{col_name}_name"  # skill_category -> skill_category_name
    code_col = f"{col_name}_code"  # skill_category -> skill_category_code
    
    # Load dictionary and normalize column names
    dict_df = pl.read_parquet(f"../data/lightcast/dictionaries/{dict_name}_dictionary.parquet")
    dict_df = dict_df.rename({col: col.lower().replace(" ", "_") for col in dict_df.columns})
    
    # Cast the join column to string to avoid type mismatch with categorical columns
    skills_calabria = skills_calabria.with_columns(pl.col(col_name).cast(pl.String))
    
    # Get original column order
    original_columns = skills_calabria.columns
    col_position = original_columns.index(col_name)
    
    # Join to add the name column
    skills_calabria = skills_calabria.join(
        dict_df.select([col_name, name_col]),
        on=col_name,
        how='left'
    ).rename({col_name: code_col})
    
    # Reorder columns: insert CODE and NAME together at original position
    current_columns = skills_calabria.columns
    other_columns = [c for c in current_columns if c not in [code_col, name_col]]
    new_order = other_columns[:col_position] + [code_col, name_col] + other_columns[col_position:]
    skills_calabria = skills_calabria.select(new_order)
    
    del dict_df

del skill_columns, original_columns, other_columns, current_columns, new_order

print(f"Skills taxonomy mapping complete. Number of rows: {skills_calabria.height:,}")

Skills taxonomy mapping complete. Number of rows: 168,923


In [12]:
# Merge skills df with OJP df

ojp_and_skills_calabria_flat = skills_calabria.join(ojp_calabria, on='id', how='left')

# Check the number of rows in DFs
ojp_ids = set(ojp_calabria.select("id").to_series().to_list())
nested_ids = set(ojp_and_skills_calabria_flat.select("id").to_series().to_list())
skills_ids = set(skills_calabria.select("id").unique().to_series().to_list())

# Find missing IDs
missing_in_nested = ojp_ids - nested_ids

print(f"IDs in ojp_calabria: {len(ojp_ids):,}")
print(f"IDs in skills_calabria: {len(skills_ids):,}")
print(f"IDs in ojp_and_skills_calabria_flat: {len(nested_ids):,}")
print(f"\nMissing IDs (in ojp but not in ojp+skills): {len(missing_in_nested):,}")

# Check if the missing postings have no skills
if len(missing_in_nested) > 0:
    print(f"\nThese {len(missing_in_nested):,} job postings have NO skills listed")
    print("The join excluded them because skills_calabria doesn't have entries for these IDs")

IDs in ojp_calabria: 26,137
IDs in skills_calabria: 22,527
IDs in ojp_and_skills_calabria_flat: 22,527

Missing IDs (in ojp but not in ojp+skills): 3,610

These 3,610 job postings have NO skills listed
The join excluded them because skills_calabria doesn't have entries for these IDs


In [13]:
# Transform the joined df to create nested dataframe of skills for each Job Posting

ojp_and_skills_calabria_nested = ojp_and_skills_calabria_flat.group_by("id").agg([
    pl.struct([
        "skill_name", 
        "skill_id", 
        "is_software", 
        "skill_category_code",
        "skill_category_name",
        "skill_subcategory_code",
        "skill_subcategory_name",
        "skill_type"
    ]).alias("skills"),
    pl.all().exclude([
        "skill_name", 
        "skill_id", 
        "is_software", 
        "skill_category_code",
        "skill_category_name",
        "skill_subcategory_code",
        "skill_subcategory_name",
        "skill_type"
    ]).first()
])

print(f"Number of rows: {ojp_and_skills_calabria_nested.height:,} (number of Calabria job postings with nested skills)")

Number of rows: 22,527 (number of Calabria job postings with nested skills)


In [16]:
ojp_and_skills_calabria_nested.write_parquet("../data/lightcast/ojp_and_skills_calabria_nested.parquet")

In [24]:
# Inspect the skills dataframe for a specific ID
id_to_find = "e63cabb068e69b23d6245f537d134aa4b2b44d63"

# Filter for the ID, explode the list, then unnest the struct
skills_unnested = (ojp_and_skills_calabria_nested
    .filter(pl.col("id") == id_to_find)
    .select("skills")
    .explode("skills")
    .unnest("skills")
)
print(f"Number of skills for this job posting ID: {skills_unnested.height}\n")
skills_unnested

Number of skills for this job posting ID: 13



skill_name,skill_id,is_software,skill_category_code,skill_category_name,skill_subcategory_code,skill_subcategory_name,skill_type
str,str,bool,str,str,str,str,cat
"""Machine Learning""","""KS1261Z68KSKR1X31KS3""",false,"""17""","""Information Technology""","""372""","""Artificial Intelligence and Ma…","""Specialized Skill"""
"""NoSQL""","""KS1272Y6RFNZKNGWDWMB""",true,"""17""","""Information Technology""","""479""","""Databases""","""Specialized Skill"""
"""Systems Integration""","""KS4415X6DKZ8BMBPY9H9""",false,"""17""","""Information Technology""","""480""","""System Design and Implementati…","""Specialized Skill"""
"""Docker (Software)""","""KSY4WFI1S164RQUBSPCC""",true,"""17""","""Information Technology""","""476""","""Software Development Tools""","""Specialized Skill"""
"""Digital Data""","""KS122YB6HR9PW5FKBJ6T""",false,"""17""","""Information Technology""","""376""","""Basic Technical Knowledge""","""Specialized Skill"""
…,…,…,…,…,…,…,…
"""CI/CD""","""ES6942CD173CAA05BA26""",false,"""17""","""Information Technology""","""474""","""Software Development""","""Specialized Skill"""
"""Amazon Web Services""","""KS120FG6YP8PQYYNQY9B""",true,"""17""","""Information Technology""","""494""","""Web Services""","""Specialized Skill"""
"""Power BI""","""KS13USA80NE38XJHA2TL""",true,"""3""","""Analysis""","""115""","""Business Intelligence Software""","""Specialized Skill"""
